# Enhanced Sharpe Ratio Inference with Jessicka Rotation

## Abstract

This notebook extends the SSRN paper "A Closed-Form Solution for Sharpe Ratio Inference under GARCH Returns" (López de Prado, Porcu, Zoonekynd, & Engle, 2026) by integrating the **Jessicka Formulation** (Samokhvalov, 2025). We demonstrate that while the SSRN framework correctly identifies variance inflation in heavy-tailed GARCH regimes, it offers no decision-theoretic remedy. By implementing **sensitivity decay**, **edge rotation**, and **overload thresholds**, we show that the Jessicka formulation reduces the sampling variance of the Sharpe estimator and improves mean performance.

**Key Contributions:**
1. Reproduction of SSRN Figure 1 (variance vs. sample size).
2. Implementation of Power-Law Decay $\sigma(n) = (1 + \beta n)^{-\eta}$ with $\eta = 1 - 2/\kappa$.
3. Demonstration of variance reduction via strategy rotation.
4. Robustness to tail index estimation.
5. Ablation study isolating components.
6. Comprehensive "Before vs. After" infographic.

**References:**
- SSRN Paper: [López de Prado et al. (2026)](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=6568702)
- Jessicka Whitepaper: [Samokhvalov (2025)](https://github.com/algomaschine/sensitivity-decay-trading/blob/main/docs/WHITEPAPER_EN.md)

In [ ]:
# =============================================================================
# 1. SETUP AND IMPORTS
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tqdm import tqdm
from matplotlib.patches import Rectangle
import os
import warnings
warnings.filterwarnings('ignore')

# Import from local functions.py (SSRN baseline)
from functions import garch_returns, estimate_parameters, formula_15, estimate_tail_index

# Create output directory
os.makedirs('outputs', exist_ok=True)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("notebook", font_scale=1.2)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10

# Random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Flags for optional heavy sections (set False for quick test)
RUN_ROBUSTNESS = True
RUN_ABLATION = True

print("Environment ready.")

In [ ]:
# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================

def standardized_student(size, df):
    if df <= 2:
        raise ValueError("Degrees of freedom must be > 2 for finite variance.")
    raw = np.random.standard_t(df, size=size)
    scaling = np.sqrt((df - 2) / df)
    return raw * scaling

def simulate_garch_paths(n_paths, T, burn_in, mu, alpha, beta, omega, nu, seed=None):
    if seed is not None:
        np.random.seed(seed)
    if alpha + beta >= 1:
        raise ValueError("GARCH process must be stationary (alpha + beta < 1)")
    sigma_uncond = np.sqrt(omega / (1 - alpha - beta))
    all_returns = []
    all_vols = []
    total_len = T + burn_in
    for _ in range(n_paths):
        innovations = standardized_student(total_len, nu)
        ret, _, var = garch_returns(
            size=total_len,
            mu=mu,
            sigma=sigma_uncond,
            alpha=alpha,
            beta=beta,
            innovations=innovations
        )
        all_returns.append(ret[burn_in:])
        all_vols.append(np.sqrt(var[burn_in:]))
    return np.array(all_returns), np.array(all_vols)

def power_law_decay(n, beta_decay, eta):
    return (1.0 + beta_decay * n) ** (-eta)

def calculate_sharpe(returns, annual_factor=252):
    if len(returns) == 0 or np.std(returns) == 0:
        return 0.0
    return (np.mean(returns) / np.std(returns, ddof=1)) * np.sqrt(annual_factor)

def calculate_max_drawdown(returns):
    if len(returns) == 0:
        return 0.0
    cum = np.cumprod(1 + np.array(returns))
    running_max = np.maximum.accumulate(cum)
    return np.min((cum - running_max) / running_max)

def apply_rotation(returns, volatilities, mu_ceiling, eta, theta, tau0, alpha_load, beta_decay=0.01, variant='full'):
    T = len(returns)
    positions = np.zeros(T)
    sigma_path = np.zeros(T)
    active_returns = []
    n_trades = 0
    mean_vol = np.mean(volatilities)
    for t in range(T):
        sigma_t = power_law_decay(n_trades, beta_decay, eta)
        sigma_path[t] = sigma_t
        if sigma_t < theta:
            positions[t] = 0
            sigma_path[t] = 0
            continue
        if variant == 'no_overload':
            tau_t = 0.0
        else:
            tau_t = tau0 * (1 + alpha_load * volatilities[t] / mean_vol)
        if np.abs(returns[t]) <= tau_t:
            positions[t] = 0
            continue
        if variant == 'no_sizing':
            pos_size = 1.0
        else:
            pos_size = sigma_t
        positions[t] = pos_size
        active_returns.append(returns[t] * pos_size)
        n_trades += 1
    return np.array(active_returns), positions, sigma_path

print("All helper functions defined.")

In [ ]:
# =============================================================================
# 3. SIMULATION PARAMETERS (Heavy-tailed GARCH)
# =============================================================================
PARAMS = {
    'mu': 0.0005,
    'omega': 1e-6,
    'alpha': 0.15,
    'beta': 0.80,
    'nu': 3.5,
    'T': 2520,
    'burn_in': 500
}
N_PATHS = 500
RANDOM_SEED = 42

TRUE_KAPPA = 3.0
TRUE_ETA = 1.0 - 2.0 / TRUE_KAPPA

THETA = 0.5
TAU0 = 0.005
ALPHA_LOAD = 0.5
BETA_DECAY = 0.005

print(f"True tail index κ ≈ {TRUE_KAPPA}")
print(f"Power-law exponent η = {TRUE_ETA:.4f}")
print(f"Rotation threshold θ = {THETA}")
print(f"Number of paths: {N_PATHS}, daily observations: {PARAMS['T']}")

In [ ]:
# =============================================================================
# 4. GENERATE GARCH PATHS
# =============================================================================
print("Simulating GARCH paths...")
np.random.seed(RANDOM_SEED)
all_returns, all_vols = simulate_garch_paths(
    n_paths=N_PATHS,
    T=PARAMS['T'],
    burn_in=PARAMS['burn_in'],
    mu=PARAMS['mu'],
    alpha=PARAMS['alpha'],
    beta=PARAMS['beta'],
    omega=PARAMS['omega'],
    nu=PARAMS['nu'],
    seed=RANDOM_SEED
)
print(f"Done. Shape: {all_returns.shape}")

In [ ]:
# =============================================================================
# 5. MAIN SIMULATION: BUY-AND-HOLD vs JESSICKA ROTATION
# =============================================================================
buy_hold_sharpes = []
rotation_sharpes = []
rotation_drawdowns = []
buy_hold_drawdowns = []
avg_positions = []
active_days_list = []
sigma_paths_all = []

print("Running strategies...")
for i in tqdm(range(N_PATHS), desc="Paths"):
    ret = all_returns[i]
    vol = all_vols[i]
    
    bh_sharpe = calculate_sharpe(ret)
    buy_hold_sharpes.append(bh_sharpe)
    buy_hold_drawdowns.append(calculate_max_drawdown(ret))
    
    mu_ceiling = np.percentile(ret[:50], 95)
    if mu_ceiling <= 0:
        mu_ceiling = np.mean(ret[:50])
    
    active_rets, positions, sigma_path = apply_rotation(
        returns=ret,
        volatilities=vol,
        mu_ceiling=mu_ceiling,
        eta=TRUE_ETA,
        theta=THETA,
        tau0=TAU0,
        alpha_load=ALPHA_LOAD,
        beta_decay=BETA_DECAY,
        variant='full'
    )
    sigma_paths_all.append(sigma_path)
    rot_sharpe = calculate_sharpe(active_rets) if len(active_rets) > 0 else 0.0
    rotation_sharpes.append(rot_sharpe)
    rotation_drawdowns.append(calculate_max_drawdown(active_rets) if len(active_rets) > 0 else 0.0)
    avg_positions.append(np.mean(positions))
    active_days_list.append(len(active_rets))

buy_hold_sharpes = np.array(buy_hold_sharpes)
rotation_sharpes = np.array(rotation_sharpes)
rotation_drawdowns = np.array(rotation_drawdowns)
buy_hold_drawdowns = np.array(buy_hold_drawdowns)
avg_positions = np.array(avg_positions)
sigma_paths_all = np.array(sigma_paths_all)

print(f"Buy-and-hold mean Sharpe: {np.mean(buy_hold_sharpes):.3f}")
print(f"Jessicka rotation mean Sharpe: {np.mean(rotation_sharpes):.3f}")
print(f"Variance reduction: {(1 - np.var(rotation_sharpes)/np.var(buy_hold_sharpes))*100:.1f}%")

In [ ]:
# =============================================================================
# 6. SSRN BASELINE: FIGURE 1 (Variance vs Sample Size)
# =============================================================================
sample_sizes = [252, 504, 1008, 2520]
sample_vars = []
theoretical_vars = []

long_innov = standardized_student(200000, PARAMS['nu'])
sigma_uncond = np.sqrt(PARAMS['omega'] / (1 - PARAMS['alpha'] - PARAMS['beta']))
long_ret, _, _ = garch_returns(
    size=200000,
    mu=PARAMS['mu'],
    sigma=sigma_uncond,
    alpha=PARAMS['alpha'],
    beta=PARAMS['beta'],
    innovations=long_innov
)
true_skew = stats.skew(long_ret)
true_kurt = stats.kurtosis(long_ret) + 3
true_SR = PARAMS['mu'] / sigma_uncond

for T in sample_sizes:
    sharpes_T = [calculate_sharpe(all_returns[i][:T]) for i in range(N_PATHS)]
    sample_vars.append(np.var(sharpes_T, ddof=1))
    V_theo = formula_15(SR=true_SR, skew=true_skew, kurtosis=true_kurt,
                        alpha=PARAMS['alpha'], beta=PARAMS['beta'], T=T)
    theoretical_vars.append(V_theo)

plt.figure(figsize=(10, 6))
plt.loglog(sample_sizes, sample_vars, 'o-', label='Sample Variance (Monte Carlo)', linewidth=2)
plt.loglog(sample_sizes, theoretical_vars, 'r--', label='Theoretical V_GARCH (True Params)', linewidth=2)
plt.xlabel('Sample Size (T)')
plt.ylabel('Variance of Sharpe Ratio')
plt.title(f'SSRN Figure 1: Sharpe Ratio Variance under GARCH (κ ≈ {TRUE_KAPPA})', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('outputs/figure_1_ssrn_baseline.png', bbox_inches='tight')
plt.show()
print("Figure 1 saved to outputs/")

In [ ]:
# =============================================================================
# 7. COMPARATIVE PANELS (A-D)
# =============================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Jessicka Formulation: Comprehensive Analysis', fontsize=16, fontweight='bold')

var_bh = np.var(buy_hold_sharpes)
var_rot = np.var(rotation_sharpes)
reduction = (var_bh - var_rot) / var_bh * 100
axes[0,0].bar(['Buy & Hold', 'Jessicka'], [var_bh, var_rot], color=['lightblue', 'orange'], alpha=0.7)
axes[0,0].set_ylabel('Variance of Sharpe Ratio')
axes[0,0].set_title(f'Panel A: Variance Reduction ({reduction:.1f}% ↓)', fontweight='bold')
axes[0,0].text(0, var_bh, f'{var_bh:.4f}', ha='center', va='bottom', fontweight='bold')
axes[0,0].text(1, var_rot, f'{var_rot:.4f}', ha='center', va='bottom', fontweight='bold')

parts = axes[0,1].violinplot([buy_hold_sharpes, rotation_sharpes], positions=[1,2], showmeans=True, showmedians=True)
parts['bodies'][0].set_facecolor('lightblue'); parts['bodies'][1].set_facecolor('orange')
axes[0,1].set_xticks([1,2]); axes[0,1].set_xticklabels(['Buy & Hold', 'Jessicka'])
axes[0,1].set_ylabel('Annualized Sharpe Ratio')
axes[0,1].set_title('Panel B: Sharpe Ratio Distribution', fontweight='bold')
axes[0,1].axhline(0, color='black', linestyle='--', linewidth=0.5)

max_n = min(sigma_paths_all.shape[1], 500)
steps = np.arange(max_n)
mean_decay = np.mean(sigma_paths_all[:, :max_n], axis=0)
p10 = np.percentile(sigma_paths_all[:, :max_n], 10, axis=0)
p90 = np.percentile(sigma_paths_all[:, :max_n], 90, axis=0)
theo_decay = power_law_decay(steps, BETA_DECAY, TRUE_ETA)
axes[1,0].plot(steps, mean_decay, 'b-', label='Empirical Mean', linewidth=2)
axes[1,0].fill_between(steps, p10, p90, color='blue', alpha=0.2, label='10th-90th percentile')
axes[1,0].plot(steps, theo_decay, 'r--', label=f'Theoretical (η={TRUE_ETA:.2f})', linewidth=2)
axes[1,0].set_xlabel('Exposure Count (n)')
axes[1,0].set_ylabel('Sensitivity σ(n)')
axes[1,0].set_title('Panel C: Power-Law Decay Curve', fontweight='bold')
axes[1,0].legend(); axes[1,0].set_ylim(0,1.1)

thetas = np.linspace(0.1, 0.9, 9)
sharpe_by_theta = []
subset = min(100, N_PATHS)
for th in tqdm(thetas, desc="Theta sweep"):
    temp = []
    for i in range(subset):
        ret = all_returns[i]; vol = all_vols[i]
        mu_c = np.percentile(ret[:50], 95)
        if mu_c <= 0: mu_c = np.mean(ret[:50])
        act, _, _ = apply_rotation(ret, vol, mu_c, TRUE_ETA, th, TAU0, ALPHA_LOAD, BETA_DECAY, variant='full')
        temp.append(calculate_sharpe(act) if len(act)>0 else 0.0)
    sharpe_by_theta.append(np.mean(temp))
axes[1,1].plot(thetas, sharpe_by_theta, 'o-', color='green', linewidth=2)
axes[1,1].axvline(THETA, color='red', linestyle='--', label=f'Current θ={THETA}')
axes[1,1].set_xlabel('Rotation Threshold θ')
axes[1,1].set_ylabel('Mean Sharpe Ratio')
axes[1,1].set_title('Panel D: Sensitivity to Rotation Threshold', fontweight='bold')
axes[1,1].legend(); axes[1,1].grid(True)

plt.tight_layout()
plt.savefig('outputs/figure_combined_panels_abcd.png', bbox_inches='tight')
plt.show()
print("Combined panels saved to outputs/")